# 🤖 CicDream — Entrenamiento de Modelo Propio

Este notebook entrena el motor de imágenes propio de Cic_IA usando:
- Los datos acumulados por usuarios de tu app
- Fine-tuning LoRA sobre Stable Diffusion v1.5
- GPU gratis de Google Colab (T4)

**Requisitos antes de ejecutar:**
- Tener al menos 50 imágenes con feedback en tu app
- Token de desarrollador de Cic_IA
- Token de HuggingFace (gratis en huggingface.co/settings/tokens)

**Tiempo estimado:** 2-4 horas en GPU T4 gratuita

In [ ]:
# ── CELDA 1: Verificar GPU ────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU disponible:')
    print(result.stdout)
else:
    print('❌ Sin GPU — ve a: Entorno de ejecución → Cambiar tipo → GPU T4')

In [ ]:
# ── CELDA 2: Instalar dependencias ────────────────────────────────────────
print('Instalando dependencias (puede tardar 3-5 minutos)...')
!pip install -q diffusers==0.25.0 transformers==4.37.0 accelerate==0.26.0
!pip install -q peft==0.7.1 datasets==2.16.0 huggingface_hub==0.20.3
!pip install -q torch torchvision xformers --index-url https://download.pytorch.org/whl/cu118
!pip install -q requests pillow
print('✅ Dependencias instaladas')

In [ ]:
# ── CELDA 3: Configuración ────────────────────────────────────────────────
# ⚠️ COMPLETA ESTOS VALORES ANTES DE CONTINUAR

CIC_IA_URL   = 'https://asitente-ia-cic-1.onrender.com'
CIC_IA_TOKEN = ''  # Tu token de desarrollador de Cic_IA
HF_TOKEN     = ''  # Token de HuggingFace (huggingface.co/settings/tokens)
HF_REPO      = ''  # Ej: 'tu-usuario/cicdream-v1'

# Verificar que están completos
errores = []
if not CIC_IA_TOKEN: errores.append('❌ CIC_IA_TOKEN está vacío')
if not HF_TOKEN:     errores.append('❌ HF_TOKEN está vacío')
if not HF_REPO:      errores.append('❌ HF_REPO está vacío')

if errores:
    for e in errores: print(e)
    print('\n⚠️  Completa los valores arriba y vuelve a ejecutar esta celda')
else:
    print('✅ Configuración completa')
    print(f'   App: {CIC_IA_URL}')
    print(f'   Repo HF: {HF_REPO}')

In [ ]:
# ── CELDA 4: Descargar dataset desde Cic_IA ──────────────────────────────
import requests, json

headers = {'Authorization': f'Bearer {CIC_IA_TOKEN}'}
r = requests.get(f'{CIC_IA_URL}/api/image/dataset/export', headers=headers)

if r.status_code == 200:
    data = r.json()
    dataset = data['dataset']
    print(f'✅ Dataset descargado:')
    print(f'   Total imágenes con feedback: {data["total"]}')
    print(f'   Listo para entrenar: {data["ready_for_training"]}')
    print(f'   Mensaje: {data["message"]}')
    
    if data['total'] < 10:
        print('\n⚠️  Muy pocas imágenes. Genera más imágenes en tu app y deja que los usuarios den feedback.')
    elif not data['ready_for_training']:
        print(f'\n⚠️  {data["message"]}')
        print('   Puedes continuar igual para probar el proceso.')
else:
    print(f'❌ Error: {r.status_code} — {r.text[:200]}')

In [ ]:
# ── CELDA 5: Preparar dataset ─────────────────────────────────────────────
import pandas as pd
from datasets import Dataset
import os

df = pd.DataFrame(dataset)
print(f'Dataset completo: {len(df)} registros')

# Filtrar solo imágenes con buena calificación (>= 3.5)
df_good = df[df['rating'] >= 3.5].reset_index(drop=True)
print(f'Imágenes de buena calidad (rating >= 3.5): {len(df_good)}')

# Si hay muy pocas buenas, usar todas
if len(df_good) < 5:
    print('⚠️  Pocas imágenes buenas, usando todo el dataset')
    df_good = df.reset_index(drop=True)

# Mostrar distribución de estilos
print('\nDistribución por estilo:')
print(df_good['style'].value_counts().to_string())

# Crear dataset de HuggingFace
hf_dataset = Dataset.from_pandas(df_good[['prompt', 'style', 'rating']])
print(f'\n✅ Dataset listo: {len(hf_dataset)} ejemplos de entrenamiento')

In [ ]:
# ── CELDA 6: Cargar modelo base ───────────────────────────────────────────
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

print('Cargando modelo base Stable Diffusion v1.5...')
print('(Esto puede tardar 5-10 minutos la primera vez)\n')

MODEL_ID = 'runwayml/stable-diffusion-v1-5'

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to('cuda')

print('✅ Modelo base cargado en GPU')

# Prueba rápida para verificar que funciona
print('\nGenerando imagen de prueba...')
test_img = pipe('a beautiful landscape', num_inference_steps=20).images[0]
test_img.save('/content/test_base.png')
print('✅ Prueba exitosa — imagen guardada como test_base.png')

from IPython.display import Image as IPImage
IPImage('/content/test_base.png', width=300)

In [ ]:
# ── CELDA 7: Fine-tuning con LoRA ─────────────────────────────────────────
from peft import LoraConfig, get_peft_model
from transformers import CLIPTokenizer
import torch.nn.functional as F

print('Configurando LoRA para fine-tuning...')

# Configurar LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['to_q', 'to_v', 'to_k', 'to_out.0'],
    lora_dropout=0.1,
    bias='none',
)

# Aplicar LoRA al UNet
unet = pipe.unet
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

print('\n✅ LoRA configurado')
print('Los parámetros entrenables son solo una fracción del modelo completo')
print('Esto permite entrenar en GPU gratuita de Colab')

In [ ]:
# ── CELDA 8: Loop de entrenamiento ────────────────────────────────────────
from torch.optim import AdamW
from diffusers import DDPMScheduler
import numpy as np

EPOCHS      = 3        # Épocas de entrenamiento
LR          = 1e-4     # Learning rate
BATCH_SIZE  = 1        # Batch size (1 para GPU T4 gratis)

optimizer   = AdamW(unet.parameters(), lr=LR)
noise_sched = DDPMScheduler.from_config(pipe.scheduler.config)
tokenizer   = pipe.tokenizer
text_encoder = pipe.text_encoder
vae          = pipe.vae

prompts = df_good['prompt'].tolist()
print(f'Entrenando con {len(prompts)} prompts por {EPOCHS} épocas...')
print('='*50)

unet.train()
losses = []

for epoch in range(EPOCHS):
    epoch_loss = 0
    for i, prompt in enumerate(prompts):
        # Tokenizar prompt
        tokens = tokenizer(
            prompt, padding='max_length',
            max_length=tokenizer.model_max_length,
            truncation=True, return_tensors='pt'
        ).input_ids.to('cuda')
        
        with torch.no_grad():
            encoder_hidden = text_encoder(tokens)[0]
        
        # Crear ruido aleatorio
        latents = torch.randn(1, 4, 64, 64, device='cuda', dtype=torch.float16)
        timesteps = torch.randint(0, noise_sched.config.num_train_timesteps, (1,), device='cuda').long()
        noise = torch.randn_like(latents)
        noisy_latents = noise_sched.add_noise(latents, noise, timesteps)
        
        # Predicción
        noise_pred = unet(noisy_latents, timesteps, encoder_hidden).sample
        loss = F.mse_loss(noise_pred.float(), noise.float())
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        if (i + 1) % 10 == 0:
            print(f'Época {epoch+1}/{EPOCHS} | Paso {i+1}/{len(prompts)} | Loss: {loss.item():.4f}')
    
    avg_loss = epoch_loss / len(prompts)
    losses.append(avg_loss)
    print(f'\n✅ Época {epoch+1} completada | Loss promedio: {avg_loss:.4f}\n')

print('='*50)
print(f'✅ Entrenamiento completado!')
print(f'Loss inicial: {losses[0]:.4f} → Loss final: {losses[-1]:.4f}')

In [ ]:
# ── CELDA 9: Guardar y subir modelo ──────────────────────────────────────
from huggingface_hub import HfApi, login
import os

# Login en HuggingFace
login(token=HF_TOKEN)
api = HfApi()

# Guardar modelo localmente
SAVE_PATH = '/content/cicdream_model'
os.makedirs(SAVE_PATH, exist_ok=True)

print('Guardando modelo...')
unet.save_pretrained(f'{SAVE_PATH}/unet')
pipe.tokenizer.save_pretrained(f'{SAVE_PATH}/tokenizer')
pipe.text_encoder.save_pretrained(f'{SAVE_PATH}/text_encoder')

# Crear info del modelo
model_card = f"""---
license: openrail
tags:
- stable-diffusion
- lora
- cicdream
---

# CicDream v1.0

Motor de generación de imágenes propio de Cic_IA.
Entrenado con LoRA sobre Stable Diffusion v1.5.

Generado automáticamente por cicdream_training.ipynb
"""

with open(f'{SAVE_PATH}/README.md', 'w') as f:
    f.write(model_card)

print(f'Subiendo modelo a HuggingFace Hub: {HF_REPO}...')
api.create_repo(repo_id=HF_REPO, exist_ok=True, private=True)
api.upload_folder(
    folder_path=SAVE_PATH,
    repo_id=HF_REPO,
    token=HF_TOKEN
)

print(f'\n✅ Modelo subido exitosamente!')
print(f'   URL: https://huggingface.co/{HF_REPO}')

In [ ]:
# ── CELDA 10: Instrucciones finales ──────────────────────────────────────
print('='*60)
print('✅ ENTRENAMIENTO COMPLETADO')
print('='*60)
print()
print('Para activar tu modelo en Cic_IA:')
print()
print('1. Ve a Render → Environment')
print('2. Agrega esta variable:')
print(f'   HF_CICDREAM_MODEL = {HF_REPO}')
print()
print('3. Render redesplegará automáticamente')
print()
print('4. CicDream usará tu modelo propio para generar imágenes')
print()
print('IMPORTANTE: Para que el modelo funcione en producción')
print('necesitas hosting con GPU o usar HuggingFace Inference API')
print('='*60)